# 00b — Stratégie ASRS sur données live DE40

**Source** : `data/dax_live_5min.csv` (généré par `00_collect_dax_live.ipynb`)

**Stratégie ASRS optimale (F1 + F2 + C4)** :
- Signal bar = 4ème candle de 5min (ouvre à **09:15 CET**)
- Range du signal bar doit être entre **10 et 55 pts**
- Entry long = signal_high + 2 / Entry short = signal_low − 2 (OCO)
- Stop = prix d'entrée opposé (risque = range + 4 pts)
- Exit = stop touché OU fin de journée à **17:30 CET**
- **Skip** : Vendredis, Janvier, Juillet, Août

## 1. Imports & chargement

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW_CSV = ROOT / "data" / "dax_live_5min.csv"
print(f"ROOT = {ROOT}")
print(f"CSV  = {RAW_CSV}")
assert RAW_CSV.exists(), f"Fichier introuvable : {RAW_CSV} — lancez d'abord 00_collect_dax_live.ipynb"

df_raw = pd.read_csv(RAW_CSV, index_col="datetime", parse_dates=True)

# S'assurer que l'index est en heure de Berlin (CET/CEST)
if df_raw.index.tz is None:
    df_raw.index = df_raw.index.tz_localize("Europe/Berlin", ambiguous="infer")
else:
    df_raw.index = df_raw.index.tz_convert("Europe/Berlin")

df_raw.sort_index(inplace=True)

print(f"\nDonnées chargées : {len(df_raw):,} bars")
print(f"Période          : {df_raw.index.min()} → {df_raw.index.max()}")
df_raw.head(3)

## 2. Isolation du signal bar (09:15 CET)

In [ ]:
# Le signal bar est la 4ème bougie de 5min : elle ouvre à 09:15 CET
signal_bars = df_raw[df_raw.index.time == pd.Timestamp("09:15").time()].copy()
signal_bars = signal_bars[signal_bars.index.dayofweek < 5]  # lundi-vendredi seulement

signal_bars["date"]  = signal_bars.index.date
signal_bars["month"] = signal_bars.index.month
signal_bars["dow"]   = signal_bars.index.dayofweek  # 0=lundi, 4=vendredi
signal_bars["range"] = signal_bars["high"] - signal_bars["low"]

print(f"Signal bars bruts (09:15 CET) : {len(signal_bars)}")
print(f"Range moyen : {signal_bars['range'].mean():.1f} pts")
signal_bars[["open","high","low","close","range"]].describe().round(1)

## 3. Application des filtres F1 + F2 + C4

In [ ]:
# F1 — Skip Vendredis
f1_mask = signal_bars["dow"] != 4

# F2 — Range signal bar : 10–55 pts
f2_mask = signal_bars["range"].between(10, 55)

# C4 — Skip Janvier (1), Juillet (7), Août (8)
c4_mask = ~signal_bars["month"].isin([1, 7, 8])

filtered = signal_bars[f1_mask & f2_mask & c4_mask].copy()

print(f"Brut              : {len(signal_bars)} jours")
print(f"Après F1 (no Fri) : {f1_mask.sum()} jours")
print(f"Après F1+F2       : {(f1_mask & f2_mask).sum()} jours")
print(f"Après F1+F2+C4    : {len(filtered)} jours  ← base de trading")

## 4. Simulation des trades

In [ ]:
def simulate_day(date, sig_high, sig_low, df_full):
    """
    Simule un trade ASRS pour un jour donné.

    Règles :
    - Long entry  = sig_high + 2  (buy-stop)
    - Short entry = sig_low  − 2  (sell-stop)
    - Stop long   = short entry price  (et vice-versa)
    - Exit EOD    = 17:30 CET (close du dernier bar)
    - Premier niveau touché = direction du trade

    Returns dict avec : direction, entry, stop, exit_price, pnl, exit_reason
    """
    long_entry  = sig_high + 2
    short_entry = sig_low  - 2
    long_stop   = short_entry   # stop du long = prix d'entrée short
    short_stop  = long_entry    # stop du short = prix d'entrée long
    eod_time    = pd.Timestamp("17:30").time()

    # Bars du jour après le signal bar jusqu'à 17:30
    day_bars = df_full[
        (df_full.index.date == date) &
        (df_full.index.time > pd.Timestamp("09:15").time()) &
        (df_full.index.time <= eod_time)
    ]

    if day_bars.empty:
        return None

    direction = None
    entry     = None

    for _, bar in day_bars.iterrows():
        if direction is None:
            # Chercher le premier déclenchement (OCO)
            hit_long  = bar["high"] >= long_entry
            hit_short = bar["low"]  <= short_entry
            if hit_long and hit_short:
                # Les deux touchés dans le même bar → écart trop faible, skip
                return None
            elif hit_long:
                direction = "long"
                entry     = long_entry
            elif hit_short:
                direction = "short"
                entry     = short_entry
            continue

        # Trade ouvert — surveiller stop
        if direction == "long":
            if bar["low"] <= long_stop:
                return {"direction": direction, "entry": entry,
                        "stop": long_stop, "exit_price": long_stop,
                        "pnl": long_stop - entry, "exit_reason": "stop"}
        else:
            if bar["high"] >= short_stop:
                return {"direction": direction, "entry": entry,
                        "stop": short_stop, "exit_price": short_stop,
                        "pnl": entry - short_stop, "exit_reason": "stop"}

    # Fin de journée — fermer au close du dernier bar
    if direction is not None and not day_bars.empty:
        eod_close = day_bars.iloc[-1]["close"]
        pnl = (eod_close - entry) if direction == "long" else (entry - eod_close)
        return {"direction": direction, "entry": entry,
                "stop": long_stop if direction == "long" else short_stop,
                "exit_price": eod_close, "pnl": pnl, "exit_reason": "eod"}

    return None  # pas d'entrée ce jour

print("Fonction simulate_day définie")

In [ ]:
results = []

for _, row in filtered.iterrows():
    trade = simulate_day(
        date      = row["date"],
        sig_high  = row["high"],
        sig_low   = row["low"],
        df_full   = df_raw,
    )
    if trade is not None:
        trade["date"]  = row["date"]
        trade["month"] = row["month"]
        trade["dow"]   = row["dow"]
        trade["range"] = row["range"]
        results.append(trade)

trades = pd.DataFrame(results)
trades["date"] = pd.to_datetime(trades["date"])
trades.set_index("date", inplace=True)
trades.sort_index(inplace=True)

print(f"Jours filtrés  : {len(filtered)}")
print(f"Trades ouverts : {len(trades)}")
trades.head()

## 5. Résultats

In [ ]:
if trades.empty:
    print("Aucun trade — données insuffisantes ou période trop courte.")
else:
    wins   = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] < 0]

    total_pnl  = trades["pnl"].sum()
    win_rate   = len(wins) / len(trades)
    avg_win    = wins["pnl"].mean()   if len(wins) > 0 else 0
    avg_loss   = losses["pnl"].mean() if len(losses) > 0 else 0
    pf         = wins["pnl"].sum() / abs(losses["pnl"].sum()) if len(losses) > 0 else float("inf")

    daily_ret  = trades["pnl"]
    sharpe     = (daily_ret.mean() / daily_ret.std() * (252 ** 0.5)) if daily_ret.std() > 0 else 0

    cum        = trades["pnl"].cumsum()
    running_max= cum.cummax()
    drawdown   = cum - running_max
    max_dd     = drawdown.min()

    print("═" * 45)
    print("  ASRS Live — DE40  (F1 + F2 + C4)")
    print("═" * 45)
    print(f"  Trades        : {len(trades)}")
    print(f"  Win Rate      : {win_rate*100:.1f}%")
    print(f"  Avg Win       : +{avg_win:.1f} pts")
    print(f"  Avg Loss      : {avg_loss:.1f} pts")
    print(f"  Profit Factor : {pf:.2f}")
    print(f"  Total PnL     : {total_pnl:+.0f} pts")
    print(f"  Sharpe        : {sharpe:.2f}")
    print(f"  Max Drawdown  : {max_dd:.0f} pts")
    print("═" * 45)

    stops = (trades["exit_reason"] == "stop").sum()
    eods  = (trades["exit_reason"] == "eod").sum()
    print(f"  Exits — Stop : {stops} ({stops/len(trades)*100:.0f}%)  |  EOD : {eods} ({eods/len(trades)*100:.0f}%)")

## 6. Visualisation

In [ ]:
if not trades.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("ASRS Live — DE40  (F1+F2+C4)", fontsize=13, fontweight="bold")

    # 1. PnL cumulé
    ax = axes[0, 0]
    cum = trades["pnl"].cumsum()
    cum.plot(ax=ax, color="steelblue", linewidth=1.5)
    ax.fill_between(cum.index, cum, 0,
                    where=(cum >= 0), alpha=0.2, color="green")
    ax.fill_between(cum.index, cum, 0,
                    where=(cum < 0),  alpha=0.2, color="red")
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_title("PnL cumulé")
    ax.set_ylabel("Points")
    ax.grid(True, alpha=0.3)

    # 2. Distribution des trades
    ax = axes[0, 1]
    trades["pnl"].hist(ax=ax, bins=30, color="coral", edgecolor="white")
    ax.axvline(0, color="black", linewidth=1)
    ax.axvline(trades["pnl"].mean(), color="steelblue",
               linewidth=1.5, linestyle="--", label=f"Moy {trades['pnl'].mean():.1f}")
    ax.set_title("Distribution PnL par trade")
    ax.set_xlabel("Points")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 3. PnL par mois
    ax = axes[1, 0]
    monthly = trades.resample("ME")["pnl"].sum()
    colors  = ["green" if v >= 0 else "red" for v in monthly]
    monthly.plot(kind="bar", ax=ax, color=colors, edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("PnL mensuel")
    ax.set_ylabel("Points")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3, axis="y")

    # 4. Drawdown
    ax = axes[1, 1]
    running_max = trades["pnl"].cumsum().cummax()
    drawdown    = trades["pnl"].cumsum() - running_max
    drawdown.plot(ax=ax, color="red", linewidth=1)
    ax.fill_between(drawdown.index, drawdown, 0, alpha=0.3, color="red")
    ax.set_title("Drawdown")
    ax.set_ylabel("Points")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 7. Derniers trades

In [ ]:
if not trades.empty:
    display_cols = ["direction", "entry", "stop", "exit_price", "pnl", "exit_reason", "range"]
    print("Derniers 20 trades :")
    print(trades[display_cols].tail(20).to_string())

## Résumé

| Filtre | Règle |
|--------|-------|
| F1 | Skip Vendredis |
| F2 | Range signal bar 10–55 pts |
| C4 | Skip Janvier, Juillet, Août |

Référence backtest long terme (2006–2026) : **Sharpe 1.23 · PF 1.27 · +9 278 pts · MaxDD −1 112 pts**